# RAG with llama-index

In [ ]:
!pip install -qU llama-index-llms-litellm
!pip install -qU llama-index-embeddings-cohere
!pip install -qU llama-index
!pip install -qU llama-index-llms-cohere
!pip install -qU matplotlib

In [ ]:
import os
from pathlib import Path
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent.react.base import ReActAgent
from llama_index.core.settings import Settings
from llama_index.llms.cohere import Cohere

# Configure Cohere LLM

In [ ]:
COHERE_API_KEY="SnW0xkwMGxrH7IjwZ9lK9y5DYSmvvDRhUYtJ4jxG"  
llm = Cohere(api_key=COHERE_API_KEY, model="command-r-plus-08-2024") 

Settings.llm = llm  # Global LLM setting for all components

# Load Documents

In [ ]:

docs_path = Path("docs/")
documents = SimpleDirectoryReader(docs_path).load_data()

# Create Index and Query Engine

In [ ]:


from llama_index.embeddings.cohere import CohereEmbedding

embed_model = CohereEmbedding(
    cohere_api_key=COHERE_API_KEY,
    model_name="embed-english-v3.0",
    input_type="search_document",
)

index = VectorStoreIndex.from_documents(
    documents=documents, embed_model=embed_model
)

query_engine = index.as_query_engine()

# Register Tool

In [ ]:
qa_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="multi_doc_search_tool",
        description=(
            "Use this tool to search finance_policy and customer_support_guidelines "
            "for questions about policies, procedures, RAG, or customer service strategy."
        )
    )
)

# Create Agent

In [ ]:

agent = ReActAgent.from_tools([qa_tool], llm=llm, verbose=False)


### Sample Questions for Interaction

```python
# Finance Policy-Based Questions
user_query = "When are monthly financial reports due?"

# Customer Support Guidelines Questions
user_query = "What is the expected response time for emails and chats?"

# Multi-Document Cross-Referencing Questions
user_query = "How does the finance team track customer support budget usage?"

# Natural Language Tasks to Trigger Reasoning
user_query = "Explain the documentation requirements for both financial and customer interactions."


In [ ]:

print("==Agentic RAG with LlamaIndex==\n")

user_query = "What is the expected response time for emails and chats?"
print(f"Query: {user_query}\n")

response = agent.chat(user_query)

print("\nFinal Response:")
print(response.response)


#### Add-on Function to display source documents

In [ ]:
def print_source_documents(agent_response):
    if not hasattr(agent_response, "sources") or not agent_response.sources:
        print("⚠️ No source documents found in the response.")
        return

    print("📄 Source Documents:\n")
    for source_index, source in enumerate(agent_response.sources, start=1):
        # Step 1: Navigate into `raw_output.source_nodes`
        if hasattr(source, "raw_output") and hasattr(source.raw_output, "source_nodes"):
            for node_index, node_with_score in enumerate(source.raw_output.source_nodes, start=1):
                node = node_with_score.node
                score = node_with_score.score
                metadata = node.metadata

                print(f"--- Source Document {source_index}.{node_index} ---")
                print(f"🗂 File Name      : {metadata.get('file_name')}")
                print(f"📁 File Path      : {metadata.get('file_path')}")
                print(f"📝 File Type      : {metadata.get('file_type')}")
                print(f"📅 Created On     : {metadata.get('creation_date')}")
                print(f"🧮 File Size      : {metadata.get('file_size')} bytes")
                print(f"📊 Similarity Score: {score:.4f}")
                print(f"\n📃 Document Content Preview:\n{node.text.strip()[:500]}...\n")
                print("-" * 60)
        else:
            print(f"⚠️ Source {source_index} does not contain `raw_output.source_nodes`.")

print_source_documents(response)